# Exercise 1B — Advanced Imaging Design Challenge

**Computer Vision for Industrial Systems**

This optional / advanced notebook extends Exercise 1.  
It is designed for students who finish early or for a second supervised session.

No machine learning is used here. The focus is still on image formation and classical computer vision:

1. quantitative imaging design,
2. spectral / channel contrast,
3. line-scan and temporal sampling,
4. homography-based rectification,
5. a simple non-ML anomaly baseline.

## Difficulty levels

- 🟢 Basic: everyone should be able to follow with hints
- 🟡 Intermediate: requires some engineering reasoning
- 🔴 Advanced: closer to a master-level challenge

In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root / "src"))

from skimage import io, img_as_float, color, transform, filters, morphology, measure
import cv2

from cvis_utils import (
    load_gray, show_images, plot_histograms,
    mm_per_pixel, pixels_per_feature, sobel_energy, laplacian_variance,
    contrast_between_regions, cnr_between_regions, weighted_gray,
    max_exposure_time_for_blur, required_pixels_for_feature, required_fov_for_pixel_count,
    robust_zscore_map, anomaly_score_from_map, iou
)

DATA = repo_root / "data"
ADV = DATA / "advanced"
print("Repository root:", repo_root)
print("Advanced data exists:", ADV.exists())

# Part D — Quantitative imaging design

We continue the running lecture scenario:

- defect size: **0.5 mm**
- target region width: **400 mm**
- horizontal pixels: **2448 px**
- moving part speed: **200 mm/s**

This part turns the lecture formulas into design decisions.

## Task D1 — Design feasibility table 🟢

For each desired number of pixels across a 0.5 mm defect, compute:

1. required **mm/px**,
2. required field of view for a 2448 px sensor,
3. whether a 400 mm field of view is feasible.

In [ ]:
feature_size_mm = 0.5
n_px = 2448
target_fov_mm = 400.0
desired_px = np.array([2, 3, 5, 10])

# TODO:
# Create a DataFrame with columns:
# desired_px_per_feature, required_mm_per_px, max_fov_mm_with_2448_px, feasible_for_400mm_fov
rows = []
for p in desired_px:
    # required_mm = ...
    # max_fov = ...
    # feasible = ...
    ...

df_design = pd.DataFrame(rows)
df_design

## Task D2 — Exposure time constraint 🟢 / 🟡

Assume:
- object speed = 200 mm/s
- current object-space resolution = 400 / 2448 mm/px
- maximum allowed blur = 1 px, 2 px, or 4 px

Compute the maximum exposure time for each blur tolerance.

In [ ]:
object_speed_mm_s = 200.0
mm_px_current = target_fov_mm / n_px
blur_tolerances_px = np.array([1, 2, 4])

# TODO:
# Create a table with blur tolerance and maximum exposure time in milliseconds.
rows = []
for b in blur_tolerances_px:
    # t_s = ...
    ...

df_exposure = pd.DataFrame(rows)
df_exposure

### Short interpretation D 🟡

Write a concise engineering interpretation:

- Is a 400 mm FoV compatible with reliable 0.5 mm defect localization?
- What happens if the allowed blur is only 1 px?
- Which design lever would you change first: FoV, focal length, exposure, illumination, or line speed?

In [ ]:
answer_D = """


"""
print(answer_D)

# Part E — Spectral and channel contrast

This part implements the lecture idea:

\[
I_c = \int E(\lambda) R(\lambda) S_c(\lambda) d\lambda
\]

Instead of doing full spectroscopy, we use a synthetic RGB / NIR-like / UV-like example and ask:

> Which channel or spectral choice gives the most robust defect evidence?

In [ ]:
rgb = img_as_float(io.imread(ADV / "spectral" / "material_rgb.png"))
nir = load_gray(ADV / "spectral" / "material_nir_like.png")
uv = load_gray(ADV / "spectral" / "material_uv_like.png")
defect_mask = load_gray(ADV / "spectral" / "material_defect_mask.png") > 0.5

show_images(
    [rgb, rgb[...,0], rgb[...,1], rgb[...,2], nir, uv, defect_mask],
    ["RGB", "R", "G", "B", "NIR-like", "UV-like", "defect mask"],
    ncols=4,
    figsize=(14, 7),
)

## Task E1 — Compute contrast and CNR by channel 🟢 / 🟡

Define a background mask as the region outside the defect.

Compute contrast and CNR for:
- R
- G
- B
- grayscale
- NIR-like
- UV-like

In [ ]:
gray = color.rgb2gray(rgb)
background_mask = ~defect_mask

channels = {
    "R": rgb[..., 0],
    "G": rgb[..., 1],
    "B": rgb[..., 2],
    "gray": gray,
    "NIR-like": nir,
    "UV-like": uv,
}

# TODO:
# For each channel, compute contrast_between_regions and cnr_between_regions.
rows = []
for name, img in channels.items():
    ...

df_contrast = pd.DataFrame(rows).sort_values("CNR", ascending=False)
df_contrast

## Task E2 — Design a synthetic filter / weighted grayscale image 🔴

Try to create a weighted RGB grayscale image that improves defect CNR.

Example:
```python
weighted = 0.8 * R + 0.1 * G + 0.1 * B
```

Search manually over some weight combinations and report the best.

In [ ]:
weight_candidates = [
    (1.0, 0.0, 0.0),
    (0.0, 1.0, 0.0),
    (0.0, 0.0, 1.0),
    (0.8, 0.1, 0.1),
    (0.6, 0.2, 0.2),
    (0.5, 0.4, 0.1),
    (0.3, 0.6, 0.1),
]

# TODO:
# Compute weighted grayscale image and CNR for each candidate.
rows = []
best_img = None
best_weights = None
best_cnr = -np.inf

for w in weight_candidates:
    ...

df_weights = pd.DataFrame(rows).sort_values("CNR", ascending=False)
df_weights

In [ ]:
# TODO: display RGB, best weighted image, and NIR-like image.
...

### Short interpretation E 🟡

Write 3–5 sentences:

- Which channel / spectral option gave the strongest evidence?
- Was RGB grayscale sufficient?
- What does this imply for choosing monochrome/color/NIR in an industrial setup?

In [ ]:
answer_E = """


"""
print(answer_E)

# Part F — Line scan and temporal sampling

Line-scan imaging captures one line at a time.  
For a moving web or conveyor, the process speed and line rate determine spatial sampling in the motion direction.

Lecture relation:

\[
\Delta y_{obj} = \frac{v_{obj}}{f_{line}}, \qquad
f_{line} \ge \frac{v_{obj}}{\Delta y_{req}}
\]

In [ ]:
web = load_gray(ADV / "line_scan" / "continuous_web_texture.png")
line_med = load_gray(ADV / "line_scan" / "line_scan_medium_rate.png")
line_under = load_gray(ADV / "line_scan" / "line_scan_under_sampled.png")

show_images(
    [web, line_med, line_under],
    ["original continuous texture", "medium line rate", "under-sampled line scan"],
    ncols=3,
    figsize=(14, 4),
)

## Task F1 — Required line rate 🟢

Assume:
- web speed: 1.5 m/s
- desired sampling in motion direction: 0.10 mm/line

Compute the required line rate.

In [ ]:
web_speed_m_s = 1.5
delta_y_req_mm = 0.10

# TODO:
# Convert speed to mm/s and compute f_line in lines/s.
web_speed_mm_s = ...
f_line_required = ...

print(f"Required line rate: {f_line_required:.0f} lines/s")

## Task F2 — Interpret under-sampling 🟡

Compare the original, medium-rate, and under-sampled line-scan images.

Use a line profile or edge metric to support your answer.

In [ ]:
row = web.shape[0] // 2

plt.figure(figsize=(12, 4))
# TODO: plot line profiles for web, line_med, line_under
...
plt.legend()
plt.grid(True, alpha=0.3)
plt.title("Line profiles across web texture")
plt.xlabel("column")
plt.ylabel("intensity")
plt.show()

print("Sobel energy original:", sobel_energy(web))
print("Sobel energy medium  :", sobel_energy(line_med))
print("Sobel energy under   :", sobel_energy(line_under))

### Short interpretation F 🟡

Write 2–4 sentences:

- What visual artifacts appear when the line rate is too low?
- What role would an encoder play in a real line-scan system?

In [ ]:
answer_F = """


"""
print(answer_F)

# Part G — Homography-based rectification and measurement

This part simulates a planar inspection target observed from an oblique viewpoint.

You will:
1. load a warped image,
2. use four fiducial points,
3. estimate a homography,
4. rectify the image,
5. compare measurement before and after rectification.

In [ ]:
warped = img_as_float(io.imread(ADV / "homography" / "planar_target_warped.png"))
rectified_reference = img_as_float(io.imread(ADV / "homography" / "planar_target_rectified.png"))

with open(ADV / "homography" / "homography_metadata.json", "r") as f:
    hmeta = json.load(f)

src_warped = np.array(list(hmeta["fiducials_warped_px"].values()), dtype=np.float32)
dst_rectified = np.array(list(hmeta["fiducials_rectified_px"].values()), dtype=np.float32)

show_images([warped, rectified_reference], ["warped image", "rectified reference"], ncols=2, figsize=(12, 5))
print("Warped fiducials (source):")
print(src_warped)
print("Rectified fiducials (destination):")
print(dst_rectified)

## Task G1 — Estimate homography and rectify image 🟢 / 🟡

Use OpenCV:

```python
H, status = cv2.findHomography(src_points, dst_points)
rectified = cv2.warpPerspective(image, H, output_size)
```

In [ ]:
output_w, output_h = hmeta["rectified_size_px"]

# TODO:
# 1. estimate homography H from src_warped to dst_rectified
# 2. warp the image to rectified coordinates
H, status = ...
rectified = ...

show_images([warped, rectified, rectified_reference], ["warped", "rectified by you", "reference"], ncols=3, figsize=(14, 5))
print("H =")
print(H)

## Task G2 — Measure object dimensions after rectification 🟡

The metadata gives an approximate object rectangle in rectified pixel coordinates.

Assume:
- rectified scale = 0.10 mm/px

Compute the physical width and height of the measurement object.

In [ ]:
mm_per_px_rectified = hmeta["assumed_mm_per_px_rectified"]
x1, y1, x2, y2 = hmeta["measurement_object_rectified_px"]

# TODO:
# Compute width_px, height_px, width_mm, height_mm.
width_px = ...
height_px = ...
width_mm = ...
height_mm = ...

print(f"Object width:  {width_px:.1f} px = {width_mm:.2f} mm")
print(f"Object height: {height_px:.1f} px = {height_mm:.2f} mm")

### Short interpretation G 🔴

Write 3–5 sentences:

- Why is measurement on the warped image risky?
- What does the homography correct?
- What does it **not** correct?

In [ ]:
answer_G = """


"""
print(answer_G)

# Part H — Simple non-ML anomaly baseline

This is not machine learning in the deep-learning sense.  
It is a classical statistical baseline:

1. estimate nominal pixel statistics from nominal training images,
2. compute a robust z-score map for each test image,
3. threshold or score the image,
4. inspect failure modes.

This prepares the later semester project.

In [ ]:
train_dir = ADV / "anomaly_baseline" / "train_nominal"
test_dir = ADV / "anomaly_baseline" / "test"
mask_dir = ADV / "anomaly_baseline" / "masks"
manifest = pd.read_csv(ADV / "anomaly_baseline" / "manifest.csv")

train_imgs = [load_gray(p) for p in sorted(train_dir.glob("*.png"))]
train_stack = np.stack(train_imgs, axis=0)

median_img = np.median(train_stack, axis=0)
mad_img = np.median(np.abs(train_stack - median_img), axis=0)

show_images([median_img, mad_img], ["nominal median image", "nominal MAD image"], ncols=2, figsize=(10, 4))

## Task H1 — Compute anomaly maps and image-level scores 🟡

For each test image:

1. compute robust z-score map,
2. compute image-level score as 99th percentile of the map,
3. store scores in a table.

In [ ]:
# TODO:
# For each row in manifest, load the test image, compute z-score map, compute score.
rows = []
score_maps = {}

for _, row in manifest.iterrows():
    ...

df_scores = pd.DataFrame(rows).sort_values("score", ascending=False)
df_scores

## Task H2 — Choose threshold from nominal scores and evaluate 🟡 / 🔴

Use only nominal test images to set a threshold.

Example:
```python
threshold = nominal_scores.mean() + 3 * nominal_scores.std()
```

Then compute:
- true positives,
- false positives,
- false negatives,
- true negatives.

In [ ]:
# TODO:
# Choose threshold from nominal scores and compute confusion counts.
nominal_scores = ...
threshold = ...

df_scores["predicted_anomaly"] = ...
tp = ...
fp = ...
fn = ...
tn = ...

print("threshold:", threshold)
print({"TP": tp, "FP": fp, "FN": fn, "TN": tn})
df_scores

## Task H3 — Visualize strongest anomaly and a false case 🔴

Display:
- highest-scoring anomaly,
- one nominal image near the threshold,
- corresponding score maps.

If there are false positives or false negatives, inspect one of them.

In [ ]:
# TODO:
# Choose images to display and visualize image + score map.
...

### Final advanced interpretation

Write a short mini-report:

1. Which design calculation was most important?
2. Which spectral/channel choice gave the best defect evidence?
3. What line-scan or motion-sampling risk did you observe?
4. Why does homography help measurement?
5. What are the limitations of the simple anomaly baseline?

In [ ]:
advanced_report = """
1. Design calculation:


2. Spectral/channel choice:


3. Line-scan / temporal sampling:


4. Homography and measurement:


5. Simple anomaly baseline:


"""
print(advanced_report)